# Week 2 — Spiking simulation and sanity checks

Uses the weight set from `02_weights.ipynb` and the LIF module `sim/lif.py` (Shiu et al. 2024 parameters, copied from their `model.py`).

Checks, in order (each must pass before the next):
1. **Direct targets:** stimulate JO-C/E; their direct postsynaptic partners should fire.
2. **Known benchmark:** sugar GRNs (LB3b/LB3c) → proboscis motor neuron MN9
3. **Your pathway:** JO-C/E → 68 descending candidates → 81 front-leg motor neurons
4. **Shuffled control:** degree-preserving rewiring of the check 2 and 3 subnetworks

Memory: WSL is capped at 4 GB, so the checks run on subnetworks. Each build prints peak memory.

### Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BRAIN = Path.home() / 'arm_simulation/brain'     # on Colab, change only these two lines
DATA = BRAIN / 'data'
sys.path.insert(0, str(BRAIN))

from sim.lif import WeightSet, LIFNet, SHIU, peak_mem_gb, shuffle_sub, raster
from brian2 import Hz, second, prefs

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
print(f"{ws.N:,} neurons, {len(ws.count):,} edges, guessed neurons: {ws.meta['guessed_neurons']:,}")
from brian2.codegen.runtime.cython_rt import CythonCodeObject
print(f"Brian2 fast C code (Cython) available: {CythonCodeObject.is_available()}")
print(f"peak memory: {peak_mem_gb():.2f} GB")

### Cell 1: check the histamine fill

Step 2 filled 5,854 unclear neurons as histamine. That's fine if they are photoreceptor types; if non-photoreceptor types show up here, tell me.

In [ ]:
h = ws.neurons[ws.neurons.guessed & (ws.neurons.nt_used == 'histamine')]
print(f"guessed as histamine: {len(h):,}")
display(h['cls'].value_counts().to_frame('neurons'))
display(h['type'].value_counts().head(15).to_frame('neurons'))

### Cell 2: neuron groups

JO-C/E and the Week 1 shortlist, plus a search for the check 2 neurons (sugar GRNs and MN9). Send me the search output; the exact male-CNS type names decide check 2.

In [ ]:
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
print(f"JO-C/E: {len(JO)} neurons (Week 1: 335)")

shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
LEG_MN  = g2i.loc[shortlist.query("group == 'output_T1_leg_MN'").bodyId].to_numpy()
print(f"descending candidates: {len(DN_cand)}  front-leg motor neurons: {len(LEG_MN)}")

types = ws.neurons['type'].astype(str)
sens = ws.neurons[(ws.neurons.cls == 'cb_sensory') &
                  types.str.contains(r'(?i)gr5|gr64|sugar|sweet|gust|GRN|taste|pharyn|labell')]
print(f"\ngustatory-looking sensory neurons: {len(sens)}")
display(sens['type'].value_counts().head(30).to_frame('neurons'))

mn = ws.neurons[types.str.contains(r'(?i)^MN9|mn9|proboscis')]
print(f"MN9-looking neurons: {len(mn)}")
display(mn[['bodyId', 'type', 'cls', 'nt_used']])

### Cell 3: check 1, direct targets

Subnetwork = JO-C/E plus every neuron they synapse onto directly. Stimulate JO-C/E at 150 Hz for 1 s (5 trials), plus a no-stimulus baseline.

**Pass:** baseline is silent (the model has no spontaneous firing), JO-C/E fire near 150 Hz, and targets with many synapses from JO-C/E fire. If strongly connected targets stay silent, it's a bug, not biology.

In [ ]:
tgt = ws.partners(JO, 'out', min_syn=1).drop(JO, errors='ignore')
print(f"direct targets of JO-C/E: {len(tgt):,}")

sub1 = ws.subnetwork(np.concatenate([JO, tgt.index]))
net1 = LIFNet(sub1, stim_global=JO)

spk0 = net1.run(rate=0 * Hz, n_trials=2)
print(f"baseline spikes: {len(spk0)} (should be 0)\n")

spk1 = net1.run(n_trials=5)
r1 = net1.rates(spk1, ws)
r1['syn_from_JO'] = tgt.reindex(r1.index)

t1 = r1.loc[tgt.index]
print(f"\nJO-C/E mean rate: {r1.loc[JO, 'rate_hz'].mean():.0f} Hz (stimulus 150 Hz)")
for k in [1, 5, 10, 20, 50]:
    s = t1[t1.syn_from_JO >= k]
    print(f"targets with >= {k:2d} synapses from JO: {len(s):5,}, firing >= 1 Hz: {(s.rate_hz >= 1).mean():6.1%}")
print(f"rank correlation, synapses from JO vs rate: {t1.syn_from_JO.corr(t1.rate_hz, method='spearman'):.2f}")
display(t1.sort_values('syn_from_JO', ascending=False).head(20))

### Cell 4: raster and rate vs connection strength

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

show = np.concatenate([JO[:40], t1.sort_values('syn_from_JO', ascending=False).index[:60]])
local = pd.Series(np.arange(len(net1.keep)), index=net1.keep).loc[show].to_numpy()
row = {l: r for r, l in enumerate(local)}
s = spk1[(spk1.trial == 0) & spk1.local.isin(local)]
ax[0].scatter(s.t, s.local.map(row), s=1, c='k')
ax[0].axhline(39.5, color='tab:red', lw=0.8)
ax[0].set(xlabel='time (s)', ylabel='neuron', title='Trial 0: 40 JO-C/E (below line), top 60 targets (above)')

ax[1].scatter(t1.syn_from_JO, t1.rate_hz, s=6, alpha=0.5)
ax[1].set(xscale='log', xlabel='synapses from JO-C/E', ylabel='rate (Hz)', title='Direct targets')
plt.tight_layout()
plt.savefig(DATA / 'check1_direct_targets.png', dpi=120)

---
### Cell 5: check 2 groups, sugar GRNs and MN9

In male-CNS, labellar taste neurons are typed by anatomy (LB…), not by taste. Tastekin, de Haan Vicente et al. 2026 assign **LB3b and LB3c** as sugar-sensing. MN9 is the pair 10331 (L) and 16949 (R), found in Cell 2.

In [ ]:
types = ws.neurons['type'].astype(str)
lb = ws.neurons[(ws.neurons.cls == 'cb_sensory') & types.str.match(r'^LB')]
print("labellar (LB) sensory types:")
display(lb['type'].value_counts().sort_index().to_frame('neurons').T)

SUGAR = ws.by_type(r'^LB3(b|c)$', cls='cb_sensory')
MN9 = ws.by_type(r'^MN9$')
assert len(SUGAR) > 0, "no LB3b/LB3c found: send me the LB type list above"
print(f"sugar GRNs: {len(SUGAR)}")
display(ws.neurons.loc[SUGAR].groupby(['type', 'side'], dropna=False).size().to_frame('neurons'))
display(ws.neurons.loc[MN9, ['bodyId', 'type', 'side', 'nt_used']])

### Cell 6: check 2 subnetwork

The full network doesn't fit in 4 GB, so this keeps neurons that lie on **short paths** from sugar GRNs to MN9: a neuron is kept if (hops from sugar) + (hops to MN9) ≤ L. The search counts only connections with ≥5 synapses; all connections among the kept neurons are simulated.

The cell starts at the shortest sugar → MN9 path length and adds one hop at a time while the subnetwork stays under ~2.5M connections.

Caveat: neurons off these paths are missing, including some inhibition, so MN9 may respond more strongly here than in the full network. The chosen `w_syn` gets confirmed on the full network on Colab later.

In [ ]:
MAX_CONN = 2_500_000

on_path, d_fwd, d_bwd = ws.path_neurons(SUGAR, MN9, max_len=6, min_syn=5)
shortest = int(d_fwd[MN9][d_fwd[MN9] >= 0].min())
print(f"\nshortest sugar -> MN9 path: {shortest} hops "
      f"(MN9 L: {d_fwd[MN9[0]]}, R: {d_fwd[MN9[1]]})\n")

total = d_fwd.astype(int) + d_bwd
ok = (d_fwd >= 0) & (d_bwd >= 0)
sizes = []
for L in range(shortest, 7):
    k = np.union1d(np.flatnonzero(ok & (total <= L)), np.concatenate([SUGAR, MN9]))
    n_conn = len(ws.subnetwork(k)['w'])
    sizes.append((L, len(k), n_conn))
    print(f"L = {L}: {len(k):>7,} neurons, {n_conn:>10,} connections")
    if n_conn > MAX_CONN:
        break

L2, _, _ = [s_ for s_ in sizes if s_[2] <= MAX_CONN][-1]
keep2 = np.union1d(np.flatnonzero(ok & (total <= L2)), np.concatenate([SUGAR, MN9]))
sub2 = ws.subnetwork(keep2)
print(f"\nusing L = {L2}: {len(keep2):,} neurons, {len(sub2['w']):,} connections")
display(ws.neurons.loc[keep2, 'cls'].value_counts().head(10).to_frame('neurons'))

### Cell 7: check 2 simulation

Stimulate all LB3b/LB3c at 150 Hz (Shiu's default) and 50 Hz, 5 trials each, plus a no-stimulus baseline.

**Pass:** baseline silent, and MN9 fires under sugar stimulation. If MN9 is silent, that alone doesn't mean a bug: `w_syn` was tuned on FlyWire and may be too weak here, which the Step 5 sweep tests. A bug is if neurons one step downstream of sugar are silent too.

In [ ]:
net2 = LIFNet(sub2, stim_global=SUGAR)
spk = net2.run(rate=0 * Hz, n_trials=1)
print(f"baseline spikes: {len(spk)} (should be 0)\n")

res2 = {}
for rate in [50, 150]:
    spk = net2.run(rate=rate * Hz, n_trials=5)
    r = net2.rates(spk, ws)
    res2[rate] = r
    print(f"  sugar GRN rate {r.loc[SUGAR, 'rate_hz'].mean():.0f} Hz | "
          f"MN9: " + ', '.join(f"{ws.neurons.side[i]} {r.rate_hz[i]:.1f} Hz" for i in MN9) +
          f" | subnetwork neurons firing: {(r.drop(SUGAR).rate_hz >= 1).mean():.1%}\n")

direct = ws.partners(SUGAR, 'out', min_syn=5).drop(SUGAR, errors='ignore')
direct = direct[direct.index.isin(keep2)]
r = res2[150]
print(f"direct sugar targets (>=5 syn) in subnetwork: {len(direct)}, firing >= 1 Hz: {(r.loc[direct.index, 'rate_hz'] >= 1).mean():.0%}")
print("\nmost active neurons at 150 Hz (excluding sugar GRNs):")
display(r.drop(SUGAR).sort_values('rate_hz', ascending=False).head(20))

### Cell 8: check 2 raster

In [ ]:
spk2 = net2.run(rate=150 * Hz, n_trials=1, verbose=False)
top2 = res2[150].drop(np.concatenate([SUGAR, MN9])).sort_values('rate_hz', ascending=False).index[:40]
fig, ax = plt.subplots(figsize=(10, 4))
raster(ax, spk2, net2, {'sugar GRN': SUGAR, 'top 40 active': top2, 'MN9': MN9})
ax.set_title('Check 2: sugar GRNs at 150 Hz')
plt.tight_layout(); plt.savefig(DATA / 'check2_sugar_MN9.png', dpi=120)

---
### Cell 9: check 3 subnetwork, JO-C/E → front-leg motor neurons

Same short-path rule as check 2, with the 68 descending candidates always included so their rates get recorded. The leg motor neurons are in the nerve cord, so expect longer paths than sugar → MN9.

In [ ]:
sub3, L3, sizes3, d_jo = ws.path_subnetwork(JO, LEG_MN, max_len=7, min_syn=5, extra=DN_cand)
keep3 = sub3['keep']
print(f"subnetwork: {len(keep3):,} neurons, {len(sub3['w']):,} connections")
print(f"leg MNs reachable from JO-C/E within {L3} hops: {(d_jo[LEG_MN] >= 0).sum()} of {len(LEG_MN)}, "
      f"hop distance: {pd.Series(d_jo[LEG_MN]).value_counts().sort_index().to_dict()}")
display(ws.neurons.loc[keep3, 'cls'].value_counts().head(10).to_frame('neurons'))

### Cell 10: check 3 simulation

**Readout question:** which of the 68 descending candidates fire, and do the front-leg motor neurons follow?

In [ ]:
net3 = LIFNet(sub3, stim_global=JO)
spk = net3.run(rate=0 * Hz, n_trials=1)
print(f"baseline spikes: {len(spk)} (should be 0)\n")

spk3 = net3.run(n_trials=5)
r3 = net3.rates(spk3, ws)

dn = shortlist.query("group == 'middle_descending'")[['bodyId', 'type', 'side', 'role', 'score']].copy()
dn['idx'] = g2i.loc[dn.bodyId].to_numpy()
dn['rate_hz'] = r3.loc[dn.idx, 'rate_hz'].to_numpy()
dn['hops_from_JO'] = d_jo[dn.idx]
dn['fired'] = dn.rate_hz >= 1

mn_r = r3.loc[LEG_MN, 'rate_hz']
all_dn = r3[r3.cls == 'descending_neuron']
print(f"JO-C/E: {r3.loc[JO, 'rate_hz'].mean():.0f} Hz")
print(f"descending candidates firing: {dn.fired.sum()} of {len(dn)}")
print(f"all descending neurons in subnetwork firing: {(all_dn.rate_hz >= 1).sum()} of {len(all_dn)}")
print(f"front-leg MNs firing: {(mn_r >= 1).sum()} of {len(LEG_MN)}, mean {mn_r.mean():.1f} Hz, max {mn_r.max():.1f} Hz")
print(f"subnetwork neurons firing: {(r3.drop(JO).rate_hz >= 1).mean():.1%}\n")

display(dn.groupby('role').agg(candidates=('fired', 'size'), fired=('fired', 'sum'), mean_rate=('rate_hz', 'mean')))
display(dn.sort_values('rate_hz', ascending=False).head(25))
print("most active front-leg MNs:")
display(r3.loc[LEG_MN].sort_values('rate_hz', ascending=False).head(10))
print("most active descending neurons not in the candidate list:")
display(all_dn.drop(dn.idx, errors='ignore').sort_values('rate_hz', ascending=False).head(10))

### Cell 11: check 3 raster and pruned descending list

In [ ]:
fired_dn = dn[dn.fired].sort_values('rate_hz', ascending=False)
fired_dn.to_csv(DATA / 'dn_pruned_week2.csv', index=False)
print(f"saved dn_pruned_week2.csv: {len(fired_dn)} descending neurons")

spk = net3.run(n_trials=1, verbose=False)
fig, ax = plt.subplots(figsize=(10, 5))
raster(ax, spk, net3, {'JO-C/E': JO, 'DN candidates': dn.sort_values('rate_hz', ascending=False).idx,
                       'front-leg MN': mn_r.sort_values(ascending=False).index})
ax.set_title('Check 3: JO-C/E at 150 Hz')
plt.tight_layout(); plt.savefig(DATA / 'check3_JO_DN_MN.png', dpi=120)

---
### Cell 12: check 4, shuffled control

Rewire each subnetwork randomly, keeping every neuron's number of inputs and outputs and each connection's weight and sign. Only *who connects to whom* changes. 3 shuffles × 5 trials each.

**Pass:** the target responses (MN9; descending candidates and leg MNs) drop sharply in the shuffled versions. That shows the specific wiring, not just the amount of excitation, produces the response.

In [ ]:
def summary2(r):
    return dict(MN9_L=r.rate_hz[MN9[0]], MN9_R=r.rate_hz[MN9[1]],
                frac_firing=(r.drop(SUGAR).rate_hz >= 1).mean())

def summary3(r):
    return dict(DN_cand_firing=int((r.loc[dn.idx, 'rate_hz'] >= 1).sum()),
                DN_cand_mean_hz=r.loc[dn.idx, 'rate_hz'].mean(),
                legMN_firing=int((r.loc[LEG_MN, 'rate_hz'] >= 1).sum()),
                legMN_mean_hz=r.loc[LEG_MN, 'rate_hz'].mean(),
                frac_firing=(r.drop(JO).rate_hz >= 1).mean())

rows = [dict(check='2 sugar→MN9', network='real', **summary2(res2[150])),
        dict(check='3 JO→DN→MN', network='real', **summary3(r3))]
for s in range(3):
    n = LIFNet(shuffle_sub(sub2, seed=s), stim_global=SUGAR, verbose=False)
    rows.append(dict(check='2 sugar→MN9', network=f'shuffled {s}', **summary2(n.rates(n.run(n_trials=5, verbose=False), ws))))
    n = LIFNet(shuffle_sub(sub3, seed=s), stim_global=JO, verbose=False)
    rows.append(dict(check='3 JO→DN→MN', network=f'shuffled {s}', **summary3(n.rates(n.run(n_trials=5, verbose=False), ws))))
    del n
    print(f"shuffle {s} done, peak memory {peak_mem_gb():.2f} GB")

shuf = pd.DataFrame(rows).sort_values(['check', 'network']).set_index(['check', 'network'])
shuf.to_csv(DATA / 'check4_shuffle.csv')
display(shuf.round(2))

### Cell 13: check 4, activity-matched shuffled control

Cell 12 showed that shuffled networks run away (about 60% of neurons firing vs 12–25% with real wiring), which makes the target comparison unfair. Here each shuffled network's `w_syn` is turned down until the **same share of neurons fires as in the real network**, by bisection on the scale. Then the target responses are compared.

**Pass:** at matched overall activity, MN9 (check 2) and the descending candidates and leg MNs (check 3) respond clearly less in the shuffled networks than in the real one.

In [ ]:
def frac_firing(net, stim, scale, trials=1):
    r = net.rates(net.run(n_trials=trials, w_scale=scale, verbose=False), ws)
    return r, (r.drop(stim).rate_hz >= 1).mean()

def matched_shuffles(sub, stim, target_frac, summ, label, n_shuffles=3):
    rows = []
    for s in range(n_shuffles):
        n = LIFNet(shuffle_sub(sub, seed=s), stim_global=stim, verbose=False)
        lo, hi = 0.02, 1.0
        for _ in range(9):                       # bisection on a log scale
            mid = (lo * hi) ** 0.5
            _, f = frac_firing(n, stim, mid)
            lo, hi = (lo, mid) if f > target_frac else (mid, hi)
        scale = (lo * hi) ** 0.5
        r, f = frac_firing(n, stim, scale, trials=5)
        rows.append(dict(check=label, network=f'shuffled {s}, matched', w_scale=round(scale, 3), **summ(r)))
        del n
        print(f"{label} shuffle {s}: w_scale {scale:.3f}, firing {f:.1%} (target {target_frac:.1%})")
    return rows

rows = [dict(check='2 sugar→MN9', network='real', w_scale=1.0, **summary2(res2[150])),
        dict(check='3 JO→DN→MN', network='real', w_scale=1.0, **summary3(r3))]
rows += matched_shuffles(sub2, SUGAR, summary2(res2[150])['frac_firing'], summary2, '2 sugar→MN9')
rows += matched_shuffles(sub3, JO, summary3(r3)['frac_firing'], summary3, '3 JO→DN→MN')

matched = pd.DataFrame(rows).set_index(['check', 'network'])
matched.to_csv(DATA / 'check4_shuffle_matched.csv')
display(matched.round(2))

---
## Step 5: `w_syn` sweep

Only `w_syn` changes. Threshold, time constants and stimulus strength stay at Shiu's values. Each scale multiplies Shiu's 0.275 mV.

At each scale, on both subnetworks (sugar at 150 Hz, JO-C/E at 150 Hz, 3 trials each):
- **target response:** MN9 L/R; descending candidates (number firing, mean rate); front-leg MNs (number firing, mean rate)
- **runaway:** share of all subnetwork neurons firing ≥1 Hz

**Selection rule, fixed before looking at the results:** the *lowest* scale at which MN9 L ≥ 20 Hz **and** the descending candidates' mean rate ≥ 20 Hz, provided neither subnetwork has more than 30% of its neurons firing. Lowest, because more weight than needed only pushes the network toward runaway.

Caveat: runaway on a subnetwork understates runaway on the full network. The chosen scale gets confirmed on the full network on Colab.

In [ ]:
SCALES = [0.25, 0.35, 0.5, 0.7, 1.0, 1.4, 2.0, 2.8, 4.0]
N_TRIALS = 3

rows = []
for sc in SCALES:
    a = summary2(net2.rates(net2.run(rate=150 * Hz, n_trials=N_TRIALS, w_scale=sc, verbose=False), ws))
    b = summary3(net3.rates(net3.run(rate=150 * Hz, n_trials=N_TRIALS, w_scale=sc, verbose=False), ws))
    rows.append(dict(w_scale=sc, w_syn_mV=round(0.275 * sc, 4),
                     MN9_L=a['MN9_L'], MN9_R=a['MN9_R'], firing_sugar_sub=a['frac_firing'],
                     DN_cand_firing=b['DN_cand_firing'], DN_cand_mean_hz=b['DN_cand_mean_hz'],
                     legMN_firing=b['legMN_firing'], legMN_mean_hz=b['legMN_mean_hz'],
                     firing_JO_sub=b['frac_firing']))
    print(f"scale {sc:4}: MN9 L {a['MN9_L']:6.1f} Hz | DN cand {b['DN_cand_mean_hz']:6.1f} Hz | "
          f"leg MN {b['legMN_firing']:2d} firing | runaway {a['frac_firing']:.0%} / {b['frac_firing']:.0%}")

sweep = pd.DataFrame(rows)
sweep.to_csv(DATA / 'wsyn_sweep_subnetworks.csv', index=False)

ok = sweep[(sweep.MN9_L >= 20) & (sweep.DN_cand_mean_hz >= 20) &
           (sweep.firing_sugar_sub <= 0.30) & (sweep.firing_JO_sub <= 0.30)]
CHOSEN = float(ok.w_scale.min()) if len(ok) else None
print(f"\nscales meeting the rule: {ok.w_scale.tolist()}")
print(f"chosen w_scale: {CHOSEN}" + (f"  (w_syn = {0.275 * CHOSEN:.4f} mV)" if CHOSEN else "  -- none: send me the table"))
display(sweep.round(2))

### Cell 15: sweep plot and saved choice

In [ ]:
import json
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].plot(sweep.w_scale, sweep.MN9_L, 'o-', label='MN9 L')
ax[0].plot(sweep.w_scale, sweep.MN9_R, 'o--', label='MN9 R')
ax[0].set(title='Check 2: sugar → MN9', ylabel='rate (Hz)')
ax[1].plot(sweep.w_scale, sweep.DN_cand_mean_hz, 'o-', label='DN candidates, mean')
ax[1].plot(sweep.w_scale, sweep.legMN_mean_hz, 'o-', label='front-leg MNs, mean')
ax[1].set(title='Check 3: JO-C/E → DN → leg MN', ylabel='rate (Hz)')
ax[2].plot(sweep.w_scale, sweep.firing_sugar_sub, 'o-', label='sugar subnetwork')
ax[2].plot(sweep.w_scale, sweep.firing_JO_sub, 'o-', label='JO subnetwork')
ax[2].axhline(0.30, color='grey', ls=':', label='30% limit')
ax[2].set(title='Runaway: share of neurons firing', ylabel='fraction ≥ 1 Hz', ylim=(0, 1))
for a in ax:
    a.set(xscale='log', xlabel='w_syn scale (× 0.275 mV)')
    a.axvline(1.0, color='grey', lw=0.8)
    if CHOSEN:
        a.axvline(CHOSEN, color='tab:red', lw=1.5, label=f'chosen {CHOSEN}×')
    a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(DATA / 'wsyn_sweep.png', dpi=120)

if CHOSEN:
    choice = dict(w_scale=CHOSEN, w_syn_mV=0.275 * CHOSEN, status='chosen on subnetworks; confirm on full network',
                  rule='lowest scale with MN9_L >= 20 Hz and DN candidate mean >= 20 Hz, both subnetworks <= 30% firing',
                  weights=str(ws.folder.name))
    (DATA / 'wsyn_choice.json').write_text(json.dumps(choice, indent=2))
    print(json.dumps(choice, indent=2))

### Cell 16: pruned descending list at the chosen scale

Re-runs check 3 at the chosen `w_syn` (10 trials) and overwrites `dn_pruned_week2.csv`. A neuron counts as fired at ≥ 1 Hz mean.

In [ ]:
assert CHOSEN, "no w_syn scale chosen in Cell 14"
r3c = net3.rates(net3.run(n_trials=10, w_scale=CHOSEN, verbose=False), ws)
dn_c = dn.drop(columns=['rate_hz', 'fired']).assign(rate_hz=r3c.loc[dn.idx, 'rate_hz'].to_numpy(),
                                                    rate_hz_at_1x=dn.rate_hz.to_numpy())
dn_c['fired'] = dn_c.rate_hz >= 1
dn_c['w_scale'] = CHOSEN
pruned = dn_c[dn_c.fired].sort_values('rate_hz', ascending=False)
pruned.to_csv(DATA / 'dn_pruned_week2.csv', index=False)
print(f"at {CHOSEN}x: {len(pruned)} of {len(dn_c)} descending candidates fired -> dn_pruned_week2.csv")
print(f"front-leg MNs firing: {(r3c.loc[LEG_MN, 'rate_hz'] >= 1).sum()} of {len(LEG_MN)}")
display(dn_c.groupby('role').agg(candidates=('fired', 'size'), fired=('fired', 'sum'), mean_rate=('rate_hz', 'mean')).round(1))
display(pruned[['type', 'side', 'role', 'hops_from_JO', 'rate_hz', 'rate_hz_at_1x']].round(1))